In [ ]:
#| default_exp study

# Study data

> Read review history and card scheduling state from a collection

Reviews done on other devices reach the collection through `sync`, as rows of the `revlog` table. This module reads them, and the scheduling state of each card, so they can be analysed outside Anki: spaced-repetition statistics, FSRS parameter fitting, dashboards.

Cards are identified by their note's **guid** and template number (`ord`), not by their id. The guid is the stable identifier a deck publisher controls, and it survives re-importing a package (see `import_apkg`), so the history stays attached to the right card whatever happens to the ids.

There is no scheduler here, so this is read-only: fastanki can't answer a card, only read what other clients recorded.

In [ ]:
#| export
import csv, json
from fastcore.utils import *
from fastanki.collection import *

In [ ]:
import json, tempfile
from fastcore.test import *

## Reviews

`revlog.id` is the time of the review in milliseconds since the epoch. `rating` is the button pressed, 1 (Again) to 4 (Easy), and 0 for a manual reschedule. `type` is 0 learning, 1 review, 2 relearning, 3 filtered, 4 manual and 5 rescheduled.

In [ ]:
#| export
_REVIEW_KEYS = ('guid', 'ord', 'time_ms', 'rating', 'type', 'ivl', 'last_ivl', 'factor', 'duration_ms', 'card_id')

@patch
def reviews(self:Collection):
    "All reviews, oldest first, as dicts keyed by `guid`, `ord`, `time_ms`, `rating`, `type`, `ivl`, `last_ivl`, `factor`, `duration_ms`, `card_id`"
    rows = self.q("""select n.guid, c.ord, r.id, r.ease, r.type, r.ivl, r.lastIvl, r.factor, r.time, r.cid
        from revlog r join cards c on c.id=r.cid join notes n on n.id=c.nid order by r.id""")
    return [dict(zip(_REVIEW_KEYS, r)) for r in rows]

## Card state

`memory` is the card's `data` column decoded: when a card has been studied with FSRS it holds the memory state (`s` stability, `d` difficulty).

In [ ]:
#| export
@patch
def card_states(self:Collection):
    "Scheduling state of every card, as dicts keyed by `guid`, `ord`, `deck`, `type`, `queue`, `due`, `ivl`, `reps`, `lapses`, `memory`"
    out = []
    for g,o,d,ty,qu,due,ivl,reps,lapses,data in self.q("""select n.guid, c.ord, d.name, c.type, c.queue, c.due, c.ivl, c.reps, c.lapses, c.data
            from cards c join notes n on n.id=c.nid join decks d on d.id=c.did order by n.guid, c.ord"""):
        try: mem = json.loads(data or '{}')
        except ValueError: mem = {}
        out.append(dict(guid=g, ord=o, deck=d.replace('\x1f','::'), type=ty, queue=qu, due=due, ivl=ivl, reps=reps, lapses=lapses, memory=mem))
    return out

## FSRS export

The optimizer for FSRS ([fsrs-rs-python](https://github.com/open-spaced-repetition/fsrs-rs-python), `fsrs-optimizer`) reads a CSV of `card_id, review_time, review_rating, review_state, review_duration`, where the state is 0 new, 1 learning, 2 review, 3 relearning. Anki's `type` 0, 1, 2 map to states 1, 2, 3; manual and rescheduled entries and filtered-deck reviews don't describe memory, so they are left out, as are entries with rating 0.

In [ ]:
#| export
_FSRS_STATE = {0:1, 1:2, 2:3}

@patch
def revlog_csv(self:Collection, path):
    "Write the review history to `path` in the format FSRS optimizers read; returns the number of rows"
    n = 0
    with open(path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['card_id', 'review_time', 'review_rating', 'review_state', 'review_duration'])
        for r in self.reviews():
            if r['type'] not in _FSRS_STATE or r['rating']==0: continue
            w.writerow([r['card_id'], r['time_ms'], r['rating'], _FSRS_STATE[r['type']], r['duration_ms']])
            n += 1
    return n

## Example

A review log, written the way another client would: rows in `revlog` for a card.

In [ ]:
tmp = Path(tempfile.mkdtemp())
col = Collection.open(tmp/'c.anki2')
n = col.add(Front='hola', Back='hello')
cid = col.q1('select id from cards where nid=?', n.id)
col.con.execute("insert into revlog values (1700000000000,?,0,3,1,1,2500,4000,0)", (cid,))
col.con.execute("insert into revlog values (1700000100000,?,0,1,0,1,2500,2000,2)", (cid,))
col.reviews()

In [ ]:
test_eq([r['rating'] for r in col.reviews()], [3,1])
test_eq(col.reviews()[0]['guid'], n.guid)
test_eq(col.card_states()[0]['deck'], 'Default')
test_eq(col.revlog_csv(tmp/'revlog.csv'), 2)
print((tmp/'revlog.csv').read_text())